# 04 - Interactive figure (Bokeh): survey speed vs. image quality

Reads the binned tables written by `01_data_and_binning.ipynb` and builds the interactive
version of the figure. Hover gives the period, nights, visits, medians and IQR on both axes,
the weather flag and a storm-truncation note. Toggles: all vs. weather-excluded nights, nightly
cloud, ROST band; a dropdown switches the image-quality statistic; a range slider limits the
periods shown.

`PERIOD` selects the binning from the same cache:

- `"window"` (default): the best two weeks of Early Ops against the first two weeks of LSST.
  The nightly points of the two windows are shown from the start.
- `"quarter"`: fiscal quarters (FY26 Q1 = Oct–Dec 2025, ...), with the storm gap drawn.
- `"month"`: calendar months (cluttered, for reference).

Output: a standalone HTML file in `figures/` (Bokeh JS inlined, about 2 MB).


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings

import pandas as pd
from bokeh.io import output_notebook, show

warnings.simplefilter("ignore", FutureWarning)
sys.path.insert(0, ".")
from ohlp import data, interactive, reference as ref

output_notebook()

In [ ]:
# Parameters (must match 01_data_and_binning.ipynb)
DAY_OBS_MIN = ref.DAY_OBS_EARLY_OPS_START
DAY_OBS_MAX = 20261014
PERIOD = "window"  # "window" (two 14-night windows), "quarter" (fiscal, Oct start) or "month"
IQ_COL = "idiq_aos_cam"  # initial y statistic; switchable in the plot
SHOW_NIGHTLY = False  # nightly cloud on at start (always on for PERIOD = "window")

TABLE_NAME = {"window": "windows", "quarter": "quarterly", "month": "monthly"}[PERIOD]
tag = f"{DAY_OBS_MIN}_{DAY_OBS_MAX}"
periods = pd.read_parquet(data.DATA_DIR / f"{TABLE_NAME}_{tag}.parquet")
nightly = pd.read_parquet(data.DATA_DIR / f"nightly_{tag}.parquet")
print(len(periods) // 2, f"{PERIOD}s,", int((periods.query('series == "all"').n_visits > 0).sum()), "with data")
periods.query("series == 'all'")[["label_long", "n_nights_calendar", "n_nights_with_visits", "n_visits", "ratio", f"{IQ_COL}_med"]].round(3)


In [ ]:
layout = interactive.build_figure(periods, nightly, iq_col=IQ_COL, show_nightly=SHOW_NIGHTLY)
show(layout)

In [ ]:
out = interactive.save_html(
    layout,
    data.DATA_DIR.parent / "figures" / f"speed_vs_iq_interactive_{TABLE_NAME}_{DAY_OBS_MIN}-{DAY_OBS_MAX}.html",
)
print(out, f"{out.stat().st_size / 1e6:.1f} MB")
